# 🎯 Workshop Day 3: Evaluation & Optimization

**Agentic RAG Workshop** | Day 3 of 3

---

### 🎯 วันนี้เราจะเรียนรู้
1. **วัดคุณภาพ RAG** — ด้วย RAGAS (4 metrics)
2. **สร้าง Eval Dataset** — อัตโนมัติด้วย Gemini
3. **ทดสอบ Agent** — Tool Selection, LLM-as-Judge
4. **ปรับปรุง Pipeline** — A/B Experiment
5. **Prompt Engineering** — เทคนิคเขียน Instruction ให้ดีขึ้น
6. **Capstone Project** — รวมทุกอย่างจาก 3 วัน ⭐

### 🗺️ ภาพรวม 3 วัน

```
Day 1 (Data)           Day 2 (Agent)          Day 3 (Evaluation)
─────────────          ─────────────          ─────────────────
Raw → Chunk →          Agent + Tool →         วัดคุณภาพ (3.1-3.2)
Embed → VectorDB       RAG Agent →            ทดสอบ Agent (3.3)
  → Retrieve           Multi-Agent →          ปรับปรุง (3.4-3.5)
                       Agentic RAG            Capstone ⭐ (3.6)
```

> 💡 วันนี้คือ "ตรวจงาน" — รู้ว่าระบบดีแค่ไหน แล้วทำให้ดีขึ้น

## 📦 Section 0: ติดตั้ง Dependencies

In [ ]:
%%time
!pip install -q ragas datasets google-adk google-genai \
    sentence-transformers qdrant-client langchain-text-splitters \
    rank_bm25 scikit-learn

import warnings
warnings.filterwarnings('ignore')
print('✅ ติดตั้งเรียบร้อย!')

In [ ]:
%%time
import os, json, hashlib, random, asyncio
import numpy as np

try:
    from google.colab import userdata
    os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
    print('✅ โหลด API Key จาก Colab Secrets')
except Exception:
    api_key = input('🔑 กรุณาวาง Gemini API Key: ')
    os.environ['GOOGLE_API_KEY'] = api_key

os.environ['GOOGLE_GENAI_USE_VERTEXAI'] = 'False'

from google import genai
try:
    client = genai.Client(api_key=os.environ['GOOGLE_API_KEY'])
    resp = client.models.generate_content(model='gemini-2.5-flash', contents='ตอบแค่ OK')
    print(f'✅ API ทำงาน: {(resp.text.strip() if resp.text else '(ไม่มี output)')}')
except Exception as e:
    print(f'❌ API Error: {e}')

### เตรียมข้อมูล + VectorDB (re-use จาก Day 1-2)

In [ ]:
%%time
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient, models
from langchain_text_splitters import RecursiveCharacterTextSplitter

embed_model = SentenceTransformer('intfloat/multilingual-e5-large')

documents = {
    'healthcare': 'โรงพยาบาลศิริราช นำ AI มาช่วยวิเคราะห์ภาพ X-ray ทรวงอก ลดเวลาวินิจฉัยจาก 30 นาทีเหลือ 5 นาที ระบบ Deep Learning ตรวจจับความผิดปกติได้แม่นยำ 95% ช่วยแพทย์คัดกรองผู้ป่วยวัณโรคได้เร็วขึ้น ลดการรอคอยผลตรวจ',
    'banking': 'ธนาคารกสิกรไทย พัฒนา KADE AI Assistant ใช้ RAG ดึงข้อมูลผลิตภัณฑ์ทางการเงินมาตอบลูกค้า ลดเวลาตอบจาก 5 นาทีเหลือ 30 วินาที รองรับภาษาไทยและอังกฤษ จัดการคำถามสินเชื่อ บัตรเครดิต การลงทุนได้อัตโนมัติ',
    'education': 'จุฬาลงกรณ์มหาวิทยาลัย สร้างระบบ RAG ถาม-ตอบจากตำราเรียน แปลง PDF กว่า 500 เล่มเป็น vector embeddings ใช้ multilingual model รองรับภาษาไทย นักศึกษาถามคำถามแล้วได้คำตอบพร้อมอ้างอิง',
    'agriculture': 'กรมวิชาการเกษตร ใช้ AI วิเคราะห์ภาพถ่ายโดรน ตรวจจับโรคพืชในนาข้าว ความแม่นยำ 92% ช่วยลดความเสียหายจากโรคพืชได้ 40% ใช้ Computer Vision และ CNN ที่ train ด้วยภาพโรคข้าว 50000 ภาพ',
    'rag': 'สถาปัตยกรรม RAG ประกอบด้วย 3 ส่วน Retriever ค้นหาจาก VectorDB Reader อ่านเอกสาร Generator สร้างคำตอบ ช่วยลดปัญหา hallucination เพราะคำตอบมาจากข้อมูลจริง ไม่ใช่ LLM แต่งเอง',
    'embedding': 'Text Embedding แปลงข้อความเป็น vector multilingual-e5-large รองรับ 100 ภาษารวมภาษาไทย สร้าง vector 1024 มิติ ข้อความที่คล้ายกันจะมี vector ใกล้กัน ใช้ Cosine Similarity วัดความคล้าย',
    'kmitl': 'สถาบันพระจอมเกล้าเจ้าคุณทหารลาดกระบัง (KMITL) คณะวิศวกรรมศาสตร์ พัฒนาระบบ AI สำหรับ Smart Campus ใช้ IoT sensor รวมกับ Machine Learning วิเคราะห์การใช้พลังงานในอาคาร ลดค่าไฟได้ 25% นอกจากนี้ยังพัฒนาระบบ NLP ภาษาไทยสำหรับวิเคราะห์ความคิดเห็นนักศึกษา และนำ RAG มาสร้างระบบ AI Tutor ช่วยตอบคำถามวิชาเรียนจากเอกสารประกอบการสอน',
    'nlp': 'NLP ภาษาไทยมีความท้าทาย ภาษาไทยไม่มีเว้นวรรคระหว่างคำ ต้องใช้ Word Segmentation เช่น PyThaiNLP สระ วรรณยุกต์ เพิ่มความซับซ้อน Tokenizer ต้องออกแบบเฉพาะ',
    'vectordb': 'Qdrant เป็น Vector Database ประสิทธิภาพสูง รองรับ ANN ค้นหาเร็วแม้มีล้าน vectors รองรับ Cosine Dot Product L2 Distance มี payload filter ค้นหาพร้อม metadata ได้',
}

splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=30)
all_chunks, all_sources = [], []
for src, text in documents.items():
    for chunk in splitter.split_text(text):
        all_chunks.append(chunk)
        all_sources.append(src)

passages = [f'passage: {c}' for c in all_chunks]
embeddings = embed_model.encode(passages, show_progress_bar=True)
VECTOR_SIZE = embeddings.shape[1]

qdrant = QdrantClient(':memory:')
qdrant.create_collection('thai_ai_kb',
    vectors_config=models.VectorParams(size=VECTOR_SIZE, distance=models.Distance.COSINE))
points = [models.PointStruct(id=i, vector=embeddings[i].tolist(),
    payload={'text': all_chunks[i], 'source': all_sources[i]}) for i in range(len(all_chunks))]
qdrant.upsert('thai_ai_kb', points=points)

print(f'✅ Data ready: {len(all_chunks)} chunks, {len(documents)} sources')

# RAG function
def search_qdrant(query, top_k=3):
    q_vec = embed_model.encode(f'query: {query}').tolist()
    results = qdrant.query_points('thai_ai_kb', query=q_vec, limit=top_k).points
    return [{'text': r.payload['text'], 'source': r.payload['source'],
             'score': round(r.score, 4)} for r in results]

def rag_answer(question, top_k=3):
    contexts = search_qdrant(question, top_k)
    context_text = '\n'.join([c['text'] for c in contexts])
    prompt = f'"""ตอบคำถามจากข้อมูลต่อไปนี้ ตอบเป็นภาษาไทย กระชับ\n\nข้อมูล:\n{context_text}\n\nคำถาม: {question}\nคำตอบ:"""'
    resp = client.models.generate_content(model='gemini-2.5-flash', contents=prompt,
        config=genai.types.GenerateContentConfig(temperature=0.3))
    return (resp.text.strip() if resp.text else '(ไม่มี output)'), contexts

# Quick test
ans, ctx = rag_answer('AI ช่วยหมอไทยยังไง')
print(f'\n🧪 Test RAG: {ans[:80]}...')

---
## 📊 Section 3.1: RAGAS — วัดคุณภาพ RAG

### RAGAS คืออะไร?

**RAGAS (Retrieval-Augmented Generation Assessment Suite)** = framework วัดคุณภาพ RAG

```
Input: question + answer + contexts (+ ground_truth)
  ↓ RAGAS คำนวณ
Output: คะแนน 0.0 - 1.0 ต่อ metric
```

### 4 Metrics หลัก

| Metric | วัดอะไร | เปรียบเทียบ | ต้อง ground_truth? |
|--------|---------|------------|:------------------:|
| **Faithfulness** | คำตอบ "ซื่อสัตย์" ต่อข้อมูลมั้ย? (ไม่หลอน?) | answer ↔ contexts | ❌ |
| **Answer Relevancy** | คำตอบตรงคำถามมั้ย? | answer ↔ question | ❌ |
| **Context Precision** | ดึง chunk มาเกี่ยวข้องจริงมั้ย? | contexts ↔ ground_truth | ✅ |
| **Context Recall** | ดึง chunk มาครบมั้ย? | contexts ↔ ground_truth | ✅ |

### 🌡️ แปลผลคะแนน

| คะแนน | ระดับ | ต้องทำอะไร |
|:-----:|-------|----------|
| 0.8-1.0 | 🟢 ดีเยี่ยม | ✅ พร้อมใช้งาน |
| 0.6-0.8 | 🟡 พอใช้ | ⚠️ ปรับปรุงได้ |
| < 0.6 | 🔴 ต้องแก้ | ❌ ต้องปรับปรุงด่วน |

In [ ]:
%%time
# ─── สร้าง Evaluation Dataset ───
# ต้องมี: question, answer, contexts, ground_truth

eval_questions = [
    'AI ช่วยวงการแพทย์ไทยอย่างไร',
    'RAG มีสถาปัตยกรรมอย่างไร',
    'ธนาคารไทยใช้ AI อย่างไร',
    'NLP ภาษาไทยมีปัญหาอะไร',
    'Qdrant ทำงานอย่างไร',
]

eval_ground_truths = [
    'โรงพยาบาลศิริราชใช้ AI วิเคราะห์ X-ray ทรวงอก ลดเวลาวินิจฉัยจาก 30 นาทีเหลือ 5 นาที ความแม่นยำ 95%',
    'RAG มี 3 ส่วน Retriever ค้นหาจาก VectorDB, Reader อ่านเอกสาร, Generator สร้างคำตอบ ลดปัญหา hallucination',
    'กสิกรไทยพัฒนา KADE AI Assistant ใช้ RAG ดึงข้อมูลผลิตภัณฑ์ ลดเวลาตอบจาก 5 นาทีเหลือ 30 วินาที',
    'ภาษาไทยไม่มีเว้นวรรคระหว่างคำ ต้องใช้ Word Segmentation เช่น PyThaiNLP สระ วรรณยุกต์เพิ่มความซับซ้อน',
    'Qdrant เป็น Vector Database ประสิทธิภาพสูง รองรับ ANN ค้นหาเร็ว รองรับ Cosine Dot Product L2',
]

# วน RAG เก็บ answers + contexts
eval_answers = []
eval_contexts = []

print('🔄 กำลังสร้าง answers...')
for q in eval_questions:
    ans, ctxs = rag_answer(q)
    eval_answers.append(ans)
    eval_contexts.append([c['text'] for c in ctxs])
    print(f'  ✅ {q[:30]}... → {ans[:40]}...')

print(f'\n📊 Eval Dataset: {len(eval_questions)} questions')

In [ ]:
%%time
# ─── วัดด้วย RAGAS ───
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
from datasets import Dataset

eval_dataset = Dataset.from_dict({
    'question': eval_questions,
    'answer': eval_answers,
    'contexts': eval_contexts,
    'ground_truth': eval_ground_truths,
})

print('🔄 กำลังวัดคุณภาพด้วย RAGAS... (อาจใช้เวลา 1-2 นาที)')
try:
    result = evaluate(
        eval_dataset,
        metrics=[faithfulness, answer_relevancy, context_precision, context_recall],
    )
    print(f'\n{"="*60}')
    print(f'📊 RAGAS Evaluation Results')
    print(f'{"="*60}')
    for metric, score in result.items():
        level = '🟢' if score >= 0.8 else ('🟡' if score >= 0.6 else '🔴')
        print(f'  {level} {metric}: {score:.4f}')
except Exception as e:
    print(f'⚠️ RAGAS error: {e}')
    print('💡 ลองใช้ manual evaluation แทน (Section 3.3)')

### 💡 สังเกต
- **Faithfulness สูง** = Agent ไม่หลอน ตอบจากข้อมูลจริง
- **Context Precision ต่ำ** = ดึง chunk ไม่เกี่ยวมา → ลองปรับ `chunk_size`, `top_k`
- **Context Recall ต่ำ** = ข้อมูลสำคัญไม่ถูกดึงมา → ลองเพิ่ม `top_k` หรือปรับ embedding

> 🎯 เป้าหมาย: ทุก metric **≥ 0.7** ถือว่าพร้อม production

### 🧪 แบบฝึกหัด 3.1
1. เพิ่มคำถาม 3 ข้อของตัวเอง + ground_truth
2. วัด RAGAS → metric ไหนต่ำสุด?
3. วิเคราะห์ว่าทำไมถึงต่ำ

In [ ]:
# 🧪 แบบฝึกหัด: เพิ่มคำถามและวัด RAGAS
# 💡 Hint:
#   1. เพิ่ม question + ground_truth เข้า list
#   2. วน rag_answer() เก็บ answer + contexts
#   3. สร้าง Dataset.from_dict() แล้ว evaluate()


---
## 🔬 Section 3.2: สร้าง Evaluation Dataset อัตโนมัติ

### ทำไมต้องสร้างอัตโนมัติ?

- เขียน ground_truth มือ → ช้ามาก (5 คำถาม = 30 นาที)
- ใช้ LLM สร้างจาก chunk → ได้ 50 คำถามใน 2 นาที

```
Chunk: "ศิริราชใช้ AI วิเคราะห์ X-ray แม่นยำ 95%"
  ↓ Gemini สร้าง
Q: "โรงพยาบาลไหนใช้ AI วิเคราะห์ภาพ X-ray?"
A: "โรงพยาบาลศิริราช ความแม่นยำ 95%"
```

In [ ]:
%%time
# ─── Auto-generate Q&A จาก chunks ───
def generate_qa_from_chunk(chunk_text):
    prompt = f'''จากข้อมูลนี้ สร้างคำถาม 1 ข้อ + คำตอบ
ข้อมูล: {chunk_text}

ตอบเป็น JSON: {{"question": "...", "answer": "..."}}
คำถามต้องเจาะจง เป็นภาษาไทย คำตอบอ้างอิงข้อมูลจริง'''
    try:
        resp = client.models.generate_content(model='gemini-2.5-flash', contents=prompt,
            config=genai.types.GenerateContentConfig(temperature=0.3, response_mime_type='application/json'))
        return json.loads(resp.text)
    except:
        return None

# สร้างจาก 8 chunks (เลือก chunk แรกของแต่ละ source)
auto_qa = []
seen_sources = set()
for i, chunk in enumerate(all_chunks):
    src = all_sources[i]
    if src in seen_sources: continue
    seen_sources.add(src)
    qa = generate_qa_from_chunk(chunk)
    if qa:
        qa['source'] = src
        auto_qa.append(qa)
        print(f'  ✅ [{src}] Q: {qa["question"][:50]}...')

print(f'\n📊 สร้างได้ {len(auto_qa)} Q&A pairs')

### 💡 สังเกต
- LLM สร้างคำถามได้เร็วกว่ามือ 10 เท่า
- แต่ต้อง **ตรวจสอบคุณภาพ** — บางข้ออาจกว้างเกินไป
- ใช้เป็น **starting point** แล้วปรับด้วยมือ

> 🎯 Production: สร้างอัตโนมัติ 100 ข้อ → ตรวจมือ → เลือก 50 ข้อที่ดี

### 🧪 แบบฝึกหัด 3.2
สร้าง auto Q&A จาก chunks ทั้งหมด แล้วตรวจคุณภาพ

In [ ]:
# 🧪 แบบฝึกหัด
# 💡 Hint: วนลูป all_chunks แล้วสร้าง Q&A ด้วย generate_qa_from_chunk()


---
## 🧪 Section 3.3: ทดสอบ Agent

### ทดสอบอะไรบ้าง?

| ประเภท | ทดสอบอะไร | ตัวอย่าง |
|--------|----------|--------|
| **Tool Selection** | เลือก tool ถูกมั้ย? | ถาม BMI → เรียก calculate_bmi |
| **Response Quality** | คำตอบดีมั้ย? | ใช้ LLM-as-Judge |
| **Edge Cases** | พังมั้ย? | คำถามแปลกๆ, หลายภาษา |
| **Guardrails** | ปฏิเสธเรื่องไม่เหมาะสม? | ถามเรื่องอันตราย |

In [ ]:
# ─── สร้าง Agent สำหรับทดสอบ ───
from google.adk.agents import Agent
from google.adk.runners import InMemoryRunner
from google.genai import types

def search_knowledge(query: str) -> dict:
    """ค้นหาข้อมูล AI ในประเทศไทย
    Args:
        query: คำถามที่ต้องการค้นหา
    """
    results = search_qdrant(query, top_k=3)
    return {'status': 'success', 'results': results}

def calculate_bmi(weight_kg: float, height_cm: float) -> dict:
    """คำนวณค่า BMI
    Args:
        weight_kg: น้ำหนักเป็นกิโลกรัม
        height_cm: ส่วนสูงเป็นเซนติเมตร
    """
    h = height_cm / 100
    bmi = weight_kg / (h ** 2)
    return {'bmi': round(bmi, 1)}

test_agent = Agent(
    name='test_agent', model='gemini-2.5-flash',
    instruction='''คุณเป็นผู้เชี่ยวชาญ AI ตอบเป็นภาษาไทย
    - ใช้ search_knowledge เมื่อถูกถามเรื่อง AI ในไทย
    - ใช้ calculate_bmi เมื่อถูกถามเรื่อง BMI
    - คำถามทั่วไป ตอบเอง
    - ห้ามตอบเรื่องอันตรายหรือผิดกฎหมาย''',
    tools=[search_knowledge, calculate_bmi]
)

async def test_agent_call(agent, msg):
    runner = InMemoryRunner(agent=agent, app_name=agent.name)
    session_id = f's_{id(agent)}_{id(msg)}'
    await runner.session_service.create_session(
        app_name=agent.name, user_id='tester', session_id=session_id
    )
    content = types.Content(role='user', parts=[types.Part.from_text(text=msg)])
    response, tools = '', []
    async for event in runner.run_async(user_id='tester', session_id=session_id, new_message=content):
        if event.content and event.content.parts:
            for p in event.content.parts:
                if p.text: response = p.text
                if p.function_call: tools.append(p.function_call.name)
    return response, tools

print('✅ Test Agent พร้อม')

In [ ]:
# ─── Test Suite: Tool Selection ───
test_cases = [
    {'input': 'น้ำหนัก 70 สูง 175 BMI?',      'expected_tool': 'calculate_bmi'},
    {'input': 'AI ช่วยหมอไทยยังไง',            'expected_tool': 'search_knowledge'},
    {'input': 'สวัสดีครับ',                     'expected_tool': None},
    {'input': 'Qdrant คืออะไร',                 'expected_tool': 'search_knowledge'},
    {'input': 'อากาศวันนี้เป็นยังไง',            'expected_tool': None},
]

print('═' * 60)
print('🧪 Test Suite: Tool Selection')
print('═' * 60)

passed = 0
for tc in test_cases:
    resp, tools = await test_agent_call(test_agent, tc['input'])
    actual_tool = tools[0] if tools else None
    ok = actual_tool == tc['expected_tool']
    passed += ok
    status = '✅' if ok else '❌'
    print(f"  {status} '{tc['input'][:25]}...' → expected={tc['expected_tool']}, got={actual_tool}")

print(f'\n📊 Pass rate: {passed}/{len(test_cases)} ({passed/len(test_cases)*100:.0f}%)')

In [ ]:
%%time
# ─── LLM-as-Judge ───
def llm_judge(question, answer, max_score=5):
    prompt = f'''ให้คะแนนคำตอบ 1-{max_score} ตามเกณฑ์:
- ถูกต้อง (ตรงกับข้อเท็จจริง)
- กระชับ (ไม่ยาวเกินไป)
- ตอบตรงคำถาม

คำถาม: {question}
คำตอบ: {answer}

ตอบ JSON: {{"score": 1-{max_score}, "reason": "เหตุผลสั้นๆ"}}'''
    try:
        resp = client.models.generate_content(model='gemini-2.5-flash', contents=prompt,
            config=genai.types.GenerateContentConfig(temperature=0.1, response_mime_type='application/json'))
        return json.loads(resp.text)
    except:
        return {'score': 0, 'reason': 'error'}

# ทดสอบ
print('═' * 60)
print('🧪 LLM-as-Judge: วัดคุณภาพคำตอบ')
print('═' * 60)

judge_questions = [
    'AI ช่วยหมอไทยยังไง',
    'RAG มีกี่ส่วน',
    'ธนาคารไทยใช้ AI อะไร',
]

total_score = 0
for q in judge_questions:
    ans, _ = rag_answer(q)
    verdict = llm_judge(q, ans)
    total_score += verdict['score']
    print(f"  {'⭐'*verdict['score']}{'☆'*(5-verdict['score'])} {verdict['score']}/5 | {q[:25]}... → {verdict['reason']}")

avg = total_score / len(judge_questions)
print(f'\n📊 Average: {avg:.1f}/5.0')

### 💡 สังเกต
- **Tool Selection test** = ตรวจว่า Agent เลือก tool ถูกมั้ย (deterministic)
- **LLM-as-Judge** = ใช้ LLM ตรวจคุณภาพคำตอบ (ไม่ต้องเขียนเกณฑ์เอง)
- ทั้งสองใช้ร่วมกัน → ครอบคลุมทั้ง "ทำงานถูก" และ "ตอบดี"

> ⚠️ LLM-as-Judge ไม่ 100% แม่นยำ — ใช้เป็น screening tool ก่อน human review

### 🧪 แบบฝึกหัด 3.3
เพิ่ม test cases 5 ข้อ รวม edge cases (ภาษาอังกฤษ, คำถามแปลกๆ)

In [ ]:
# 🧪 แบบฝึกหัด
# 💡 Hint: เพิ่ม test_cases ลิสต์ แล้ววน test_agent_call()


---
## ⚡ Section 3.4: ปรับปรุง RAG Pipeline

### ปรับอะไรได้บ้าง?

| ขั้นตอน | Parameter | ผลกระทบ |
|---------|----------|----------|
| Chunking | `chunk_size`, `overlap` | Context Precision |
| Retrieval | `top_k`, `alpha` | Context Recall |
| Generation | `temperature`, prompt | Faithfulness, Relevancy |

### A/B Experiment
```
Config A: chunk=100, top_k=3
Config B: chunk=200, top_k=5  ← ดีกว่า?
Config C: chunk=300, top_k=3
```

In [ ]:
%%time
# ─── A/B Experiment ───
configs = [
    {'name': 'A: small chunks', 'chunk_size': 100, 'overlap': 20, 'top_k': 3},
    {'name': 'B: medium chunks', 'chunk_size': 200, 'overlap': 30, 'top_k': 5},
    {'name': 'C: large chunks', 'chunk_size': 300, 'overlap': 50, 'top_k': 3},
]

test_q = 'AI ช่วยวงการแพทย์ไทยอย่างไร'
test_gt = 'โรงพยาบาลศิริราชใช้ AI วิเคราะห์ X-ray แม่นยำ 95%'

print('═' * 60)
print('🧪 A/B Experiment: เปรียบเทียบ Config')
print('═' * 60)

for cfg in configs:
    sp = RecursiveCharacterTextSplitter(chunk_size=cfg['chunk_size'], chunk_overlap=cfg['overlap'])
    chunks = []
    for text in documents.values():
        chunks.extend(sp.split_text(text))
    
    # Embed + search
    vecs = embed_model.encode([f'passage: {c}' for c in chunks])
    q_vec = embed_model.encode(f'query: {test_q}')
    from sklearn.metrics.pairwise import cosine_similarity
    sims = cosine_similarity([q_vec], vecs)[0]
    top_idx = np.argsort(sims)[-cfg['top_k']:][::-1]
    top_chunks = [chunks[i] for i in top_idx]
    top_scores = [sims[i] for i in top_idx]
    
    # Check if ground truth info is in retrieved chunks
    gt_found = any('ศิริราช' in c or '95%' in c for c in top_chunks)
    
    print(f'\n📋 {cfg["name"]}: {len(chunks)} chunks')
    print(f'   Top scores: {[round(s,3) for s in top_scores]}')
    print(f'   Ground truth found: {"✅" if gt_found else "❌"}')
    print(f'   Best chunk: {top_chunks[0][:60]}...')

### 💡 สังเกต
- **chunk เล็ก** (100) → chunks มากขึ้น แต่อาจตัดบริบทสำคัญ
- **chunk ใหญ่** (300) → บริบทครบ แต่อาจมี noise มากขึ้น
- **top_k มาก** → recall สูงขึ้น แต่ precision อาจลดลง

> 🎯 ไม่มี config ที่ดีสุดสำหรับทุกงาน — ต้อง **ทดลอง** กับข้อมูลจริง

### 🧪 แบบฝึกหัด 3.4
ทดลองอย่างน้อย 3 configs → หา config ที่ดีที่สุดสำหรับข้อมูลนี้

In [ ]:
# 🧪 แบบฝึกหัด: ทดลอง configs ต่างๆ
# 💡 Hint: เพิ่ม configs ใน list แล้ววนทดสอบ


---
## ✍️ Section 3.5: Prompt Engineering สำหรับ Agent

### เทคนิค

| เทคนิค | ตัวอย่าง | ลด Hallucination? |
|--------|---------|:-----------------:|
| **Role** | "คุณเป็นผู้เชี่ยวชาญ..." | ⭐ |
| **Few-shot** | ให้ตัวอย่าง Q&A | ⭐⭐ |
| **Chain-of-Thought** | "คิดทีละขั้น..." | ⭐⭐⭐ |
| **Guardrails** | "ห้ามเดา ถ้าไม่รู้บอกตรงๆ" | ⭐⭐⭐ |
| **Output Format** | "ตอบเป็น bullet 3 ข้อ" | ⭐ |

In [ ]:
# ─── Before vs After Prompt ───
prompt_before = 'ตอบคำถาม'
prompt_after = '''คุณเป็นผู้เชี่ยวชาญ AI ในประเทศไทย ตอบเป็นภาษาไทย

กฎ:
1. ค้นหาข้อมูลจาก search_knowledge ก่อนตอบเสมอ
2. ตอบเฉพาะข้อมูลที่พบ ห้ามเดาหรือแต่งเพิ่ม
3. อ้างอิงแหล่งที่มา เช่น [healthcare] [banking]
4. ถ้าไม่พบข้อมูล บอกตรงๆ ว่า "ไม่มีข้อมูลในฐานความรู้"
5. ตอบกระชับ เป็น bullet points ไม่เกิน 5 ข้อ'''

agent_v1 = Agent(name='v1', model='gemini-2.5-flash', instruction=prompt_before, tools=[search_knowledge])
agent_v2 = Agent(name='v2', model='gemini-2.5-flash', instruction=prompt_after, tools=[search_knowledge])

test_q = 'AI ช่วยวงการแพทย์ไทยอย่างไร'

print('═' * 60)
print('🧪 Before vs After Prompt')
print('═' * 60)

r1, _ = await test_agent_call(agent_v1, test_q)
r2, _ = await test_agent_call(agent_v2, test_q)

j1 = llm_judge(test_q, r1)
j2 = llm_judge(test_q, r2)

print(f'\n📋 V1 (instruction สั้น): {j1["score"]}/5 — {j1["reason"]}')
print(f'📋 V2 (instruction ดี):   {j2["score"]}/5 — {j2["reason"]}')
print(f'\n--- V1 ---\n{r1[:200]}')
print(f'\n--- V2 ---\n{r2[:200]}')

### 💡 สังเกต
- Instruction ที่ดี → คำตอบ **อ้างอิงข้อมูล, กระชับ, มีโครงสร้าง**
- Instruction สั้น → คำตอบอาจ **เดา, ยาวเกินไป, ไม่มี reference**
- **Guardrails** ("ห้ามเดา") ลด hallucination ได้มาก

> 🎯 Prompt Engineering = วิธีปรับปรุงที่ **ง่ายที่สุด แต่ผลกระทบมากที่สุด**

### 🧪 แบบฝึกหัด 3.5
เขียน instruction 3 เวอร์ชัน → วัดด้วย LLM-as-Judge → หาเวอร์ชันที่ดีที่สุด

In [ ]:
# 🧪 แบบฝึกหัด
# 💡 Hint: สร้าง agent_v3 ด้วย instruction ใหม่ แล้ววัดเทียบ


---
## 🚀 Section 3.6: Capstone Project ⭐

### รวมทุกอย่างจาก 3 วัน

```
Day 1: เตรียมข้อมูล → Chunk → Embed → VectorDB
Day 2: สร้าง Agent → Tools → RAG Agent → Multi-Agent
Day 3: วัด RAGAS → ทดสอบ → ปรับปรุง → วัดอีกครั้ง
```

### โจทย์
1. **สร้าง RAG Pipeline** จากข้อมูลที่กำหนด
2. **วัดคุณภาพ** ด้วย RAGAS + LLM-as-Judge
3. **ปรับปรุง** ให้ได้ผลดีขึ้น (chunk_size, prompt, top_k)
4. **แสดงผล Before/After**

In [ ]:
# ─── Capstone: Before → Ater ───
print('═' * 60)
print('🚀 Capstone: ปรับปรุง RAG Pipeline')
print('═' * 60)

# Step 1: วัด baseline
print('\n📊 Step 1: Baseline')
baseline_scores = []
for q in eval_questions[:3]:
    ans, _ = rag_answer(q)
    verdict = llm_judge(q, ans)
    baseline_scores.append(verdict['score'])
baseline_avg = sum(baseline_scores) / len(baseline_scores)
print(f'   Baseline Score: {baseline_avg:.1f}/5.0')

# Step 2: ปรับปรุง (ตัวอย่าง: prompt ดีขึ้น)
print('\n📊 Step 2: After Optimization')
improved_scores = []
for q in eval_questions[:3]:
    r, _ = await test_agent_call(agent_v2, q)
    verdict = llm_judge(q, r)
    improved_scores.append(verdict['score'])
improved_avg = sum(improved_scores) / len(improved_scores)
print(f'   Improved Score: {improved_avg:.1f}/5.0')

# Step 3: แสดงผล
print(f'\n{"═"*60}')
print(f'📊 สรุป Before → After')
print(f'{"═"*60}')
print(f'  Baseline:  {"⭐" * round(baseline_avg)} {baseline_avg:.1f}/5.0')
print(f'  Improved:  {"⭐" * round(improved_avg)} {improved_avg:.1f}/5.0')
diff = improved_avg - baseline_avg
if diff > 0:
    print(f'  📈 +{diff:.1f} ดีขึ้น!')
elif diff < 0:
    print(f'  📉 {diff:.1f} แย่ลง — ต้องกลับไปใช้ baseline')
else:
    print(f'  ➡️ เท่าเดิม — ลองปรับอย่างอื่น')

### 💡 สังเกต
- **Prompt Engineering** มักเป็นวิธีที่ปรับแล้วเห็นผลเร็วที่สุด
- **Chunk size** ต้องทดลอง — ไม่มีสูตรสำเร็จ
- **วัดก่อน-หลัง** สำคัญมาก — ถ้าไม่วัดก็ไม่รู้ว่าดีขึ้นจริงมั้ย

> 🎯 "What gets measured gets improved" — ถ้าวัดไม่ได้ก็ปรับปรุงไม่ได้

### 🧪 แบบฝึกหัด 3.6
1. ปรับปรุงอย่างน้อย 2 อย่าง (prompt + chunk_size หรือ top_k)
2. วัดผล Before/After
3. อธิบายว่าทำไมดีขึ้นหรือไม่ดีขึ้น

In [ ]:
# 🧪 Capstone: ปรับปรุงด้วยตัวเอง
# 💡 Hint:
#   1. เปลี่ยน chunk_size → re-embed → re-search → วัดอีกครั้ง
#   2. เปลี่ยน instruction → สร้าง agent ใหม่ → วัดอีกครั้ง
#   3. เปลี่ยน top_k → rag_answer( ,top_k=5) → วัดอีกครั้ง


---
## 🎯 สรุป: สิ่งที่เราเรียนรู้ใน 3 วัน

| Day | เนื้อหา | เครื่องมือหลัก |
|-----|--------|-------------|
| **Day 1** | Data Engineering Pipeline | Qdrant, E5-large, BM25 |
| **Day 2** | Agent Building | Google ADK, Tools, Multi-Agent |
| **Day 3** | Evaluation & Optimization | RAGAS, LLM-as-Judge |

### Skills ที่ได้
- ✅ สร้าง RAG Pipeline ตั้งแต่ต้นจนจบ
- ✅ สร้าง Agent ที่ตัดสินใจเองได้
- ✅ วัดคุณภาพ + ปรับปรุงอย่างเป็นระบบ
- ✅ ใช้ภาษาไทยได้ตลอดทาง

### 🛣️ ไปต่ออย่างไร?
- **Production**: Deploy บน Cloud (GCP, AWS) + Database session
- **Advanced**: Fine-tune embedding สำหรับ domain เฉพาะ
- **Scale**: Qdrant Cloud + horizontal scaling
- **Monitor**: Log + alert เมื่อคุณภาพตก

---

**🙏 ขอบคุณที่เรียนด้วยกันทั้ง 3 วัน!**

> "The best way to learn AI is to build with AI."